# Summarizing a document bigger than the LLM's context window

In [9]:
with open("./moby_dick.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

In [10]:
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
import getpass
from langchain_nvidia_ai_endpoints import  ChatNVIDIA
import os


In [11]:
llm = ChatNVIDIA(
    model="mistralai/devstral-2-123b-instruct-2512",  # stable model
    api_key=os.getenv("NVIDIA_MISTRAL_API_KEY"),
)

In [ ]:
# Split :  breaking down the text into smaller chunks
text_chunks_chain = (
    RunnableLambda(lambda x: 
        [
            {
                'chunk': text_chunk, 
            }
            # splitting the text into smaller chunks: chunk_size=3000, chunk_overlap=100
            for text_chunk in 
               TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)

In [ ]:
# Map : summarizing each chunk
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()        
        }
    )
)

In [ ]:
# Reduce : summarizing the summaries into a final summary
summarize_summaries_prompt_template = """
Write a coincise summary of the following text, which joins several summaries, and include the main details.
Text: {summaries}
"""

summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)
summarize_reduce_chain = (
    RunnableLambda(lambda x: 
        {
            'summaries': '\n'.join([i['summary'] for i in x]), 
        })
    | summarize_summaries_prompt 
    | llm 
    | StrOutputParser()
)

In [15]:
map_reduce_chain = (
   text_chunks_chain
   | summarize_map_chain.map()
   | summarize_reduce_chain
)     

In [16]:
summary = map_reduce_chain.invoke(moby_dick_book)

In [17]:
print(summary)

**Concise Summary of the Excerpt from *Moby-Dick***

The excerpt follows Ishmael, the narrator, as he arrives in New Bedford seeking lodging before embarking on a whaling voyage. With little money, he rejects expensive inns and settles at the dingy *Spouter-Inn*, where he is required to share a bed. Initially troubled by this, Ishmael grows increasingly uneasy when he encounters his roommate—a tattooed harpooneer named Queequeg, whose appearance evokes primal fear. However, after an awkward confrontation and reassurance, Ishmael overcomes his prejudice and accepts Queequeg, finding a strange sense of comfort and even humanity in their shared space.

The passage not only showcases Ishmael’s character—his reflexive judgment, existential musings, and eventual openness—but also introduces Queequeg as a complex figure who defies cultural stereotypes. The scene highlights themes of **fear of the unknown, cultural misunderstanding, and human connection**, all while setting the stage for their

# Summarizing across documents

In [19]:
from langchain_community.document_loaders import WikipediaLoader

wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [ ]:
# here we are importing the libraries important for importing the documents
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader

# loaded the documents
word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

In [ ]:
# created a document list
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

In [24]:
# Map : summarizing each chunk
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)

doc_summary_chain = doc_summary_prompt | llm

In [25]:
refine_summary_template = """
Your must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far: {current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful, 
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [26]:
def refine_summary(docs):

    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = \
           {"current_refined_summary": current_refined_summary, 
            "text": doc.page_content}
        intermediate_steps.append(intermediate_step)
        
        current_refined_summary = refine_chain.invoke(intermediate_step)
        
    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

In [27]:
full_summary = refine_summary(all_docs)
print(full_summary)

{'final_summary': 'Here is the **final updated summary**, incorporating the most relevant new information from the additional document while maintaining the existing structure and clarity:\n\n---\n\n### **Final Summary: Paestum [Revised with New Information]**\n\nPaestum was a major ancient Greek city on the coast of the Tyrrhenian Sea in Magna Graecia, renowned for its exceptionally well-preserved Doric temples dating from 550 to 450 BCE. Originally founded around **600 BCE by settlers from Sybaris** as **Poseidonia**, the city thrived as an autonomous Greek polis for about 200 years, becoming a flourishing settlement by 540 BCE. It was enclosed by **defensive walls with four gates** (built in phases) and featured an **agora with Greek civic structures**, including a possible **bouleuterion** and a **heroon** (a memorial linked to the city’s founder). The three iconic Doric temples—the **Temple of Hera I (the "Basilica," 6th century BCE)**, **the Temple of Hera II (the "Temple of Nept